### Round 1 barcode distribution across wells

In [ ]:
import glob
import pandas as pd


In [ ]:

well_map = pd.read_csv("./well-counts/well-map.tsv", sep="\t")
mapping = dict(well_map[['barcode', 'Well']].values)

glob_patterns = [
    "./well-counts/*PKR-432*",
    "./well-counts/*PKR-433*",
    "./well-counts/*PKR-434*",
    "./well-counts/*PKR-435*",
    "./well-counts/*PKR-436*"
]

all_files = []
for pattern in glob_patterns:
    all_files.extend(glob.glob(pattern))

df_list = []

for file in all_files:
    df = pd.read_csv(file, header=None, sep=r"\s+")
    df.columns = ["count", "barcode"]
    
    # Keep only barcodes from well map
    df = df[df["barcode"].isin(well_map["barcode"])]
    
    df['well'] = df["barcode"].map(mapping)
    
    # Force all wells to be present:
    wells_df = pd.DataFrame({'well': well_map['Well'], 'barcode': well_map['barcode']})
    df = pd.merge(wells_df, df, on=['well', 'barcode'], how='left')
    df["count"] = df["count"].fillna(0).astype(int)
    
    sample_name = file.split("/")[-1].replace(".counts.txt", "")
    df['sample'] = sample_name
    
    df_list.append(df)

final_df = pd.concat(df_list, ignore_index=True)

final_df = final_df.sort_values(["sample", "well"])

final_df.head()

